# GAN for Synthetic AML Transaction Records

This notebook demonstrates how to train a Generative Adversarial Network (GAN) to generate synthetic Anti-Money Laundering (AML) transaction records. The synthetic data can be used for prototyping and model training.

## Import Required Libraries

Import libraries such as numpy, pandas, TensorFlow or PyTorch, and any visualization tools needed for data exploration and GAN training.

In [1]:
import numpy as np
import pandas as pd
import matplotlib.pyplot as plt
import seaborn as sns
import tensorflow as tf
from tensorflow.keras import layers, models, optimizers
from sklearn.preprocessing import LabelEncoder, MinMaxScaler
from sklearn.model_selection import train_test_split

## Define Transaction Data Schema

Specify the schema for the synthetic transaction dataset, including fields: origin_account, destination_account, amount, timestamp, transaction_type, location, and device_id.

In [2]:
transaction_schema = {
    "origin_account": "string",
    "destination_account": "string",
    "amount": "float",
    "timestamp": "datetime",
    "transaction_type": "category",
    "location": "category",
    "device_id": "string"
}

print("Transaction Data Schema:")
for field, dtype in transaction_schema.items():
    print(f"{field}: {dtype}")

Transaction Data Schema:
origin_account: string
destination_account: string
amount: float
timestamp: datetime
transaction_type: category
location: category
device_id: string


## Generate Base Synthetic Transaction Dataset

Create a function to generate a base synthetic dataset with realistic distributions for each field, simulating typical AML transaction data.

In [3]:
import random
from datetime import datetime, timedelta

def generate_synthetic_transactions(n_samples=10000):
    np.random.seed(42)
    random.seed(42)

    origin_accounts = [f"ACC{str(i).zfill(6)}" for i in np.random.randint(100000, 999999, n_samples)]
    destination_accounts = [f"ACC{str(i).zfill(6)}" for i in np.random.randint(100000, 999999, n_samples)]
    amounts = np.round(np.random.exponential(scale=2000, size=n_samples), 2)
    start_date = datetime(2022, 1, 1)
    timestamps = [start_date + timedelta(minutes=int(np.random.uniform(0, 525600))) for _ in range(n_samples)]
    transaction_types = np.random.choice(['transfer', 'payment', 'withdrawal', 'deposit'], n_samples, p=[0.5, 0.2, 0.2, 0.1])
    locations = np.random.choice(['NY', 'LA', 'CHI', 'HOU', 'MIA'], n_samples)
    device_ids = [f"DEV{str(i).zfill(5)}" for i in np.random.randint(10000, 99999, n_samples)]

    df = pd.DataFrame({
        "origin_account": origin_accounts,
        "destination_account": destination_accounts,
        "amount": amounts,
        "timestamp": timestamps,
        "transaction_type": transaction_types,
        "location": locations,
        "device_id": device_ids
    })
    return df

df = generate_synthetic_transactions(10000)
df.head()

,origin_account,destination_account,amount,timestamp,transaction_type,location,device_id
0,ACC221958,ACC724152,2871.88,2022-05-17 14:32:00,transfer,NY,DEV69142
1,ACC771155,ACC248620,1121.86,2022-11-19 16:50:00,transfer,HOU,DEV80724
2,ACC231932,ACC508294,111.80,2022-09-01 16:42:00,payment,LA,DEV72574
3,ACC465838,ACC498635,369.10,2022-03-16 23:21:00,withdrawal,MIA,DEV47734
4,ACC359178,ACC266562,1217.33,2022-04-27 18:21:00,withdrawal,MIA,DEV84997


## Preprocess Data for GAN Training

Encode categorical variables, normalize numerical fields, and split the data into training and validation sets suitable for GAN input.

In [ ]:
# Encode categorical variables
le_transaction_type = LabelEncoder()
le_location = LabelEncoder()

df['transaction_type_enc'] = le_transaction_type.fit_transform(df['transaction_type'])
df['location_enc'] = le_location.fit_transform(df['location'])

# Normalize amount
scaler_amount = MinMaxScaler()
df['amount_norm'] = scaler_amount.fit_transform(df[['amount']])

# Convert timestamp to numeric (e.g., minutes since start)
df['timestamp_num'] = (df['timestamp'] - df['timestamp'].min()).dt.total_seconds() / 60
scaler_time = MinMaxScaler()
df['timestamp_norm'] = scaler_time.fit_transform(df[['timestamp_num']])

# Select features for GAN
gan_features = ['amount_norm', 'timestamp_norm', 'transaction_type_enc', 'location_enc']
X = df[gan_features].values

# Split into train and validation
X_train, X_val = train_test_split(X, test_size=0.2, random_state=42)
print("Training shape:", X_train.shape)
print("Validation shape:", X_val.shape)

## Build GAN Architecture

Define the generator and discriminator neural network architectures tailored for tabular transaction data.

In [ ]:
latent_dim = 32
data_dim = X_train.shape[1]

def build_generator(latent_dim, data_dim):
    model = models.Sequential([
        layers.Dense(64, activation='relu', input_dim=latent_dim),
        layers.Dense(128, activation='relu'),
        layers.Dense(data_dim, activation='linear')
    ])
    return model

def build_discriminator(data_dim):
    model = models.Sequential([
        layers.Dense(128, activation='relu', input_dim=data_dim),
        layers.Dense(64, activation='relu'),
        layers.Dense(1, activation='sigmoid')
    ])
    return model

generator = build_generator(latent_dim, data_dim)
discriminator = build_discriminator(data_dim)
discriminator.compile(optimizer=optimizers.Adam(0.0002), loss='binary_crossentropy', metrics=['accuracy'])

## Train the GAN Model

Implement the GAN training loop, monitor losses, and periodically generate samples to assess training progress.

In [ ]:
epochs = 3000
batch_size = 128
half_batch = batch_size // 2

# Combined model (stacked generator and discriminator)
discriminator.trainable = False
gan_input = layers.Input(shape=(latent_dim,))
generated_data = generator(gan_input)
gan_output = discriminator(generated_data)
gan = models.Model(gan_input, gan_output)
gan.compile(optimizer=optimizers.Adam(0.0002), loss='binary_crossentropy')

d_losses, g_losses = [], []

for epoch in range(epochs):
    # Train discriminator
    idx = np.random.randint(0, X_train.shape[0], half_batch)
    real_samples = X_train[idx]
    noise = np.random.normal(0, 1, (half_batch, latent_dim))
    fake_samples = generator.predict(noise, verbose=0)

    d_loss_real = discriminator.train_on_batch(real_samples, np.ones((half_batch, 1)))
    d_loss_fake = discriminator.train_on_batch(fake_samples, np.zeros((half_batch, 1)))
    d_loss = 0.5 * np.add(d_loss_real, d_loss_fake)

    # Train generator
    noise = np.random.normal(0, 1, (batch_size, latent_dim))
    valid_y = np.ones((batch_size, 1))
    g_loss = gan.train_on_batch(noise, valid_y)

    d_losses.append(d_loss[0])
    g_losses.append(g_loss)

    if epoch % 500 == 0 or epoch == epochs - 1:
        print(f"Epoch {epoch} | D loss: {d_loss[0]:.4f} | G loss: {g_loss:.4f}")

# Plot losses
plt.figure(figsize=(10,5))
plt.plot(d_losses, label='Discriminator Loss')
plt.plot(g_losses, label='Generator Loss')
plt.legend()
plt.xlabel('Epoch')
plt.ylabel('Loss')
plt.title('GAN Training Losses')
plt.show()

## Generate Synthetic Transactions

Use the trained generator to produce new synthetic transaction records and inverse-transform them to the original schema.

In [ ]:
# Generate synthetic samples
n_gen = 5000
noise = np.random.normal(0, 1, (n_gen, latent_dim))
gen_samples = generator.predict(noise, verbose=0)

# Inverse transforms
amount_gen = scaler_amount.inverse_transform(gen_samples[:, [0]])
timestamp_gen = scaler_time.inverse_transform(gen_samples[:, [1]])
transaction_type_gen = le_transaction_type.inverse_transform(np.clip(np.round(gen_samples[:, 2]), 0, len(le_transaction_type.classes_)-1).astype(int))
location_gen = le_location.inverse_transform(np.clip(np.round(gen_samples[:, 3]), 0, len(le_location.classes_)-1).astype(int))

# Reconstruct DataFrame
gen_df = pd.DataFrame({
    "amount": amount_gen.flatten(),
    "timestamp": [df['timestamp'].min() + timedelta(minutes=float(m)) for m in timestamp_gen.flatten()],
    "transaction_type": transaction_type_gen,
    "location": location_gen
})

gen_df.head()

## Evaluate Synthetic Data Quality

Compare distributions and summary statistics of synthetic vs. real data, and visualize results to assess the realism of generated transactions.

In [ ]:
# Compare distributions
plt.figure(figsize=(12,5))
plt.subplot(1,2,1)
sns.histplot(df['amount'], color='blue', label='Real', stat='density', bins=50, alpha=0.5)
sns.histplot(gen_df['amount'], color='red', label='Synthetic', stat='density', bins=50, alpha=0.5)
plt.legend()
plt.title('Amount Distribution')

plt.subplot(1,2,2)
sns.histplot(df['transaction_type'], color='blue', label='Real', stat='probability', discrete=True)
sns.histplot(gen_df['transaction_type'], color='red', label='Synthetic', stat='probability', discrete=True)
plt.legend()
plt.title('Transaction Type Distribution')
plt.tight_layout()
plt.show()

print("Real Data Summary:")
print(df[['amount', 'transaction_type', 'location']].describe(include='all'))
print("\nSynthetic Data Summary:")
print(gen_df[['amount', 'transaction_type', 'location']].describe(include='all'))